In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install -q "pandas==2.2.2" "numpy>=2.0,<2.6" "httpx==0.28.1" "fsspec==2025.3.0" "huggingface-hub>=1.2.0,<2.0" "datasets==5.0.0" "python-dotenv==1.2.2" "langchain-core==1.4.8" "langchain-qdrant==1.1.0" "langchain-text-splitters==1.1.2" "langchain-ollama==1.1.0" "ragas==0.4.3" "sentence-transformers==5.6.0" "transformers==5.13.0"

In [2]:
!pip install langchain_google_vertexai langchain_google_genai

import sys
from types import ModuleType
import langchain_google_vertexai

# Симулируем старый модуль chat_models.vertexai
if "langchain_community.chat_models.vertexai" not in sys.modules:
    mock_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
    mock_chat_vertex.ChatVertexAI = langchain_google_vertexai.ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = mock_chat_vertex

# Симулируем старый модуль llms
if "langchain_community.llms" not in sys.modules:
    mock_llms = ModuleType("langchain_community.llms")
    mock_llms.VertexAI = langchain_google_vertexai.VertexAI
    sys.modules["langchain_community.llms"] = mock_llms

# ---- ТВОЙ ИСХОДНЫЙ КОД ТЕПЕРЬ СРАБОТАЕТ ----
from ragas.testset import TestsetGenerator
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
!sudo apt-get install zstd
!sudo apt-get update && sudo apt-get install -y zstd pciutils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 129 not upgraded.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading packag

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [25]:
!OLLAMA_KEEP_ALIVE=30m nohup ollama serve > ollama.log 2>&1 &

In [26]:
!curl http://localhost:11434

Ollama is running

In [ ]:
!ollama stop qwen2.5-coder:7b

In [8]:
!ollama pull qwen2.5-coder:7b

In [23]:
%%bash
time curl http://localhost:11434/api/generate \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen2.5-coder:7b",
    "prompt": "Кратко перескажи сюжет TES IV: Oblivion.",
    "stream": false
  }'

{"model":"qwen2.5-coder:7b","created_at":"2026-07-06T23:53:44.82306911Z","response":"\"TES IV: Oblivion\" — это игра в жанре отrole-playing (RPG), основанная на популярном серии игр \"The Elder Scrolls\". Главный герой становится новым принцем пустыне, которого должны остановить неприятных насекомых-гуманистов. Игра рассказывает историю через многочисленные подзаголовки, включающие фантастические эпизоды, приключения на дикой природе и сражения с монстрами. В процессе игрок выбирает свою роль и пути развития персонажа, что делает каждую игру уникальной. \"Oblivion\" известна своей графикой, музыкальной оркестрацией и сложной системой боя.","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,26338,80559,47099,27016,22484,143847,1802,5409,11916,16964,8178,350,1570,16824,25,55775,344,290,13,151645,198,151644,77091,198,1,28484,16824,25,55775,344,290,1,1959,67879,70522,22127,5805,572

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2673    0  2537  100   136     79      4  0:00:34  0:00:32  0:00:02   531

real	0m32.054s
user	0m0.005s
sys	0m0.008s


In [ ]:
load_dotenv("/content/drive/MyDrive/podlivion/.env")
print("QDRANT_URL =", repr(os.getenv("QDRANT_URL")))
print("QDRANT_API_KEY =", repr(os.getenv("QDRANT_API_KEY")[:10] if os.getenv("QDRANT_API_KEY") else None))

QDRANT_URL = 'https://b50da76c-c016-433d-b4dd-93f354851313.europe-west3-0.gcp.cloud.qdrant.io'
QDRANT_API_KEY = 'eyJhbGciOi'


In [ ]:
import gc
import itertools
import json
import os
from datetime import datetime, timezone
from typing import Sequence

import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_ollama import ChatOllama
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas import evaluate
from ragas.metrics import context_precision, context_recall
from ragas.run_config import RunConfig
from sentence_transformers import CrossEncoder, SentenceTransformer


load_dotenv("/content/drive/MyDrive/podlivion/.env")

INPUT_FILE_PATH = "/content/drive/MyDrive/podlivion/UESP_Wiki_Dataset/jsonl/Oblivion_Cleaned_fixed.jsonl"
GOLDEN_SET_PATH = "/content/drive/MyDrive/podlivion/UESP_Wiki_Dataset/jsonl/oblivion_golden_set.csv"
OUTPUT_PATH = "/content/drive/MyDrive/podlivion/reranker_eval_results.jsonl"
RECREATE_EMBEDDINGS = False

COLLECTION_NAME = "oblivion_lore"
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 124
RETRIEVE_K = 20
BATCH_SIZE = 400
EVAL_SAMPLE_SIZE = 20
RERANK_TOP_N_VALUES = [3, 5, 7]

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
CROSS_ENCODER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
MODEL_DEVICE = os.getenv(
    "RAG_MODEL_DEVICE",
    "cuda" if torch.cuda.is_available() else "cpu",
)

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")


class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str, device: str):
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return embeddings.tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]


class RerankingRetriever:
    def __init__(
        self,
        vector_store: QdrantVectorStore,
        cross_encoder: CrossEncoder,
        top_n: int,
    ):
        self.vector_store = vector_store
        self.cross_encoder = cross_encoder
        self.top_n = top_n

    def retrieve(self, query: str) -> list[Document]:
        documents = self.vector_store.similarity_search(query, k=RETRIEVE_K)
        if not documents:
            return []

        pairs = [(query, document.page_content) for document in documents]
        scores = self.cross_encoder.predict(pairs)
        scored_documents = sorted(
            zip(documents, scores, strict=True),
            key=lambda item: float(item[1]),
            reverse=True,
        )
        return [document for document, _ in scored_documents[: self.top_n]]


embeddings = SentenceTransformerEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    device="cpu",
)

cross_encoder = CrossEncoder(
    model_name=CROSS_ENCODER_MODEL_NAME,
    device="cpu",
)

eval_llm = ChatOllama(
    model="qwen2.5-coder:7b",
    temperature=0.0,
)

judge_llm = ChatOllama(
    model="qwen2.5-coder:7b",
    temperature=0.0,
    timeout=3600,
    num_ctx=4096,
)


def load_raw_documents(input_file_path: str) -> list[Document]:
    raw_documents = []
    with open(input_file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            metadata = data["metadata"]
            for section_title, section_text in data["clean_text"].items():
                chunk_metadata = metadata.copy()
                chunk_metadata["section_title"] = section_title
                raw_documents.append(
                    Document(page_content=section_text, metadata=chunk_metadata)
                )
    return raw_documents


def upload_documents_to_qdrant(documents: list[Document]) -> QdrantVectorStore:
    print(f"Создание коллекции {COLLECTION_NAME} в Qdrant Cloud с перезаписью...")
    vector_store = QdrantVectorStore.from_documents(
        documents=[documents[0]],
        embedding=embeddings,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        collection_name=COLLECTION_NAME,
        force_recreate=True,
    )

    print(f"Загрузка {len(documents)} чанков батчами по {BATCH_SIZE}...")
    for i in range(1, len(documents), BATCH_SIZE):
        batch = documents[i : i + BATCH_SIZE]
        vector_store.add_documents(batch)
        print(f"Загружено {min(i + BATCH_SIZE, len(documents))} / {len(documents)} чанков")

    print("Загрузка в Qdrant Cloud завершена.")
    return vector_store


def build_vector_store() -> QdrantVectorStore:
    if RECREATE_EMBEDDINGS:
        raw_documents = load_raw_documents(INPUT_FILE_PATH)
        print(f"Всего исходных секций: {len(raw_documents)}")

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        documents = text_splitter.split_documents(raw_documents)
        print(
            f"Чанков после сплита "
            f"(size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}): {len(documents)}"
        )

        return upload_documents_to_qdrant(documents)

    print(f"Подключение к существующей коллекции {COLLECTION_NAME} в Qdrant Cloud")
    return QdrantVectorStore.from_existing_collection(
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )


def create_reranker_retriever(vector_store: QdrantVectorStore, top_n: int) -> RerankingRetriever:
    return RerankingRetriever(
        vector_store=vector_store,
        cross_encoder=cross_encoder,
        top_n=top_n,
    )


def format_context(documents: Sequence[Document]) -> str:
    if not documents:
        return "Контекст не найден."
    return "\n\n".join(
        f"Фрагмент {idx}:\n{document.page_content}"
        for idx, document in enumerate(documents, start=1)
    )


def get_message_text(message) -> str:
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            part.get("text", str(part)) if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content)


def generate_answer(question: str, documents: Sequence[Document]) -> str:
    system_prompt = (
        "Ты эксперт по вселенной The Elder Scrolls IV: Oblivion. "
        "Используй только приведенные фрагменты лора, чтобы ответить на вопрос. "
        "Если ответа нет в контексте, честно скажи, что не знаешь. "
        "Не выдумывай информацию.\n\n"
        f"Контекст:\n{format_context(documents)}"
    )
    response = eval_llm.invoke(
        [
            ("system", system_prompt),
            ("human", question),
        ]
    )
    return get_message_text(response)


def generate_rag_answers(retriever: RerankingRetriever, golden_set_df: pd.DataFrame) -> pd.DataFrame:
    llm_answers = []
    total = len(golden_set_df)

    for i, (_, row) in enumerate(golden_set_df.iterrows(), start=1):
        question = row["user_input"]
        reference = row["reference"]

        print(f"[{i}/{total}] Retrieving contexts...")
        documents = retriever.retrieve(question)
        contexts = [document.page_content for document in documents]

        print(f"[{i}/{total}] Generating answer with LLM...")
        answer = generate_answer(question, documents)

        llm_answers.append(
            {
                "user_input": question,
                "response": answer,
                "retrieved_contexts": contexts,
                "reference": reference,
            }
        )
        print(f"[{i}/{total}] Done.")

    return pd.DataFrame(llm_answers)


def evaluate_rag_metrics(results: pd.DataFrame) -> dict:
    dataset = Dataset.from_dict(
        {
            "user_input": results["user_input"].tolist(),
            "response": results["response"].tolist(),
            "retrieved_contexts": results["retrieved_contexts"].tolist(),
            "reference": results["reference"].tolist(),
        }
    )
    config = RunConfig(timeout=3600, max_workers=1, max_retries=3)

    metrics_results = evaluate(
        dataset=dataset,
        metrics=[context_precision, context_recall],
        llm=judge_llm,
        embeddings=embeddings,
        run_config=config,
        raise_exceptions=True,
    )
    return {
        "context_precision": float(metrics_results["context_precision"]),
        "context_recall": float(metrics_results["context_recall"]),
    }


def print_metrics(params: dict, metrics: dict) -> None:
    print("\n--- Метрики эксперимента ---")
    print(f"Параметры: {params}")
    print(f"  context_precision: {metrics['context_precision']:.4f}")
    print(f"  context_recall:    {metrics['context_recall']:.4f}")


def save_experiment_result(params: dict, metrics: dict) -> None:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "params": params,
        "metrics": metrics,
    }
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def rag_tuning_pipeline() -> None:
    vector_store = build_vector_store()

    full_golden_set_df = pd.read_csv(GOLDEN_SET_PATH)
    golden_set_df = full_golden_set_df.sample(n=EVAL_SAMPLE_SIZE, random_state=42)

    param_grid = {
        "top_n": RERANK_TOP_N_VALUES,
    }

    keys, values = zip(*param_grid.items())
    experiments = [dict(zip(keys, v, strict=True)) for v in itertools.product(*values)]

    best_score = 0.0
    best_params = {}
    all_results = []

    for idx, params in enumerate(experiments):
        print(f"\n--- Эксперимент {idx + 1}/{len(experiments)} ---")
        experiment_params = {
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "retrieve_k": RETRIEVE_K,
            **params,
        }
        print(f"Параметры: {experiment_params}")

        retriever = create_reranker_retriever(vector_store, top_n=params["top_n"])

        results_df = generate_rag_answers(retriever, golden_set_df)
        metrics = evaluate_rag_metrics(results_df)

        print_metrics(experiment_params, metrics)
        save_experiment_result(experiment_params, metrics)
        print(f"Результаты сохранены в {OUTPUT_PATH}")

        all_results.append({"params": experiment_params, "metrics": metrics})

        if metrics["context_recall"] > best_score:
            best_score = metrics["context_recall"]
            best_params = experiment_params

        del retriever
        del results_df
        gc.collect()

    print("\n=== Итоги grid search ===")
    print(f"Лучшие параметры: {best_params}")
    print(f"Лучший context_recall: {best_score:.4f}")

    for result in all_results:
        print_metrics(result["params"], result["metrics"])


if __name__ == "__main__":
    rag_tuning_pipeline()


/tmp/ipykernel_3794/601039821.py:18: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/tmp/ipykernel_3794/601039821.py:18: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Подключение к существующей коллекции oblivion_lore в Qdrant Cloud

--- Эксперимент 1/3 ---
Параметры: {'chunk_size': 1024, 'chunk_overlap': 124, 'retrieve_k': 20, 'top_n': 3}
[1/20] Retrieving contexts...
[1/20] Generating answer with LLM...
[1/20] Done.
[2/20] Retrieving contexts...
[2/20] Generating answer with LLM...
[2/20] Done.
[3/20] Retrieving contexts...
[3/20] Generating answer with LLM...
[3/20] Done.
[4/20] Retrieving contexts...
[4/20] Generating answer with LLM...
[4/20] Done.
[5/20] Retrieving contexts...
[5/20] Generating answer with LLM...
[5/20] Done.
[6/20] Retrieving contexts...
[6/20] Generating answer with LLM...
[6/20] Done.
[7/20] Retrieving contexts...
[7/20] Generating answer with LLM...
[7/20] Done.
[8/20] Retrieving contexts...
[8/20] Generating answer with LLM...
[8/20] Done.
[9/20] Retrieving contexts...
[9/20] Generating answer with LLM...
[9/20] Done.
[10/20] Retrieving contexts...
